# KeePassXC Store (050) Demo

Parse a KeePassXC XML export and fetch requested secrets in memory.


In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

from ciphercache.store.keepassxc import KeePassXCParser


In [2]:
export_path = Path("../testdata/demopasswords.export.xml")
raw = export_path.read_text(encoding="utf-8")
parser = KeePassXCParser()
entries = parser.parse(raw)
list(entries.keys())


['demoentry1', 'demoentry2', 'Demokey in a group']

In [3]:
entries["demoentry1"]


{'password': 'demopassword',
 'title': 'demoentry1',
 'url': 'http://www.x.com',
 'username': 'demouser',
 'tags': ['tag1', 'tag2', 'some', 'tags']}

In [ ]:
# Live export (requires YubiKey + password).
# Example:
# /Applications/KeePassXC_2.7.6.app/Contents/MacOS/keepassxc-cli export --format xml \
#   --yubikey 1:12345678 testdata/demopasswords.kdbx
#
# You can capture stdout and parse it directly:
# raw = subprocess.check_output([...], text=True)
# parser.parse(raw)


### Direct keepassxc-cli export (runs in the notebook kernel)

This cell **runs keepassxc-cli inside the Jupyter kernel**, so the password/YubiKey
prompt appears in the notebook output. Use this only for direct parsing tests.


In [ ]:
# Direct keepassxc-cli export (prompts appear in this notebook).
# Adjust keepassxc_cli_path and yubikey_slot to your environment.
from ciphercache.store.keepassxc import KeePassXCClient, KeePassXCConfig

config = KeePassXCConfig(
    database_path=Path("../testdata/demopasswords.kdbx"),
    yubikey_slot="1:12345678",
    keepassxc_cli_path=Path("/Applications/KeePassXC_2.7.6.app/Contents/MacOS/keepassxc-cli"),
)
raw = KeePassXCClient(config).export_xml()
parser.parse(raw)


Passwort zum Entsperren von ../testdata/demopasswords.kdbx eingeben:

Note: If a client disconnects before the daemon sends a response, the daemon will log a disconnect and continue. This avoids crashes during unlock prompts or notebook interruptions.


In [ ]:
# Daemon unlock-on-request (prompts appear in the daemon terminal).
# Start the daemon in a shell first, e.g.:
#   uv run python scripts/run_daemon.py \
#     --db-path ../testdata/demopasswords.kdbx \
#     --yubikey auto \
#     --keepassxc-cli-path /Applications/KeePassXC_2.7.6.app/Contents/MacOS/keepassxc-cli
# (You can replace --yubikey auto with --yubikey 1:12345678.)

from ciphercache.client import Client, ClientConfig

client = Client(config=ClientConfig())
# If unlock prompts take longer, override unlock_timeout_seconds:
# config = ClientConfig(unlock_timeout_seconds=600)
# client = Client(config=config)
client.status()
client.unlock("1h", ["demoentry1"])
client.get_secret("demoentry1")


In [5]:
# Unlock-all demo (requires daemon started with --unlock-all-on-start).
from ciphercache.client import Client, ClientConfig

client = Client(config=ClientConfig())
client.status()
client.get_secret("demoentry1")


{'password': 'demopassword',
 'title': 'demoentry1',
 'url': 'http://www.x.com',
 'username': 'demouser',
 'tags': ['tag1', 'tag2', 'some', 'tags']}

In [6]:
client.close_store()

True